# Phase 3 - Hyperparameter Optimization

In [31]:
# Public Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import sys

# Custom Imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from evaluate import *
from model import *
# from train import *

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Load Training and Test Data

### Via np export (Comment out if not used)

In [32]:
# Define paths for data and models
data_dir = Path("../data/processed")

X_train_path = data_dir / "X_train.npy"
X_test_path = data_dir / "X_test.npy"
y_train_path = data_dir / "y_train.npy"
y_test_path = data_dir / "y_test.npy"

# Load the data
X_train = np.load(X_train_path)
X_test = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

# Display the first 5 rows of the training and testing sets
print("Train X Set:")
display(X_train[:5])

print("Train y Set:")
display(y_train[:5])

print("Test X Set:")
display(X_test[:5])

print("Test y Set:")
display(y_test[:5])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("X_train dtype:", X_train.dtype)
print("X_test dtype:", X_test.dtype)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Training labels:")
print(np.unique(y_train, return_counts=True))
print("Testing labels:")
print(np.unique(y_test, return_counts=True))


Train X Set:


array([[-0.14556059, -0.02652612, -0.09209785,  0.        ,  0.        ,
         0.        , -0.05334714, -0.01006743,  0.5103131 , -0.00853813,
        -0.0166307 , -0.01152596, -0.01329284, -0.02856498, -0.02246034,
        -0.06486421,  0.        ,  0.        , -0.06325226, -0.10105179,
        -0.04223455, -0.06070573, -0.07336041, -0.23893514, -0.24189419,
         0.15489504, -0.15379085,  0.11912997,  1.1243894 ,  0.6013209 ,
         0.49691516, -0.29926488, -0.47052544, -0.50515455, -0.07702747,
        -0.07003175, -0.25253636, -0.25093088,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0. 

Train y Set:


array([0, 0, 0, 0, 0])

Test X Set:


array([[-1.4556059e-01, -3.5006717e-02, -9.8555885e-02,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00, -5.3347137e-02, -1.0067435e-02,
        -1.9595814e+00, -8.5381325e-03, -1.6630700e-02, -1.1525964e-02,
        -1.3292844e-02, -2.8564978e-02, -2.2460341e-02, -6.4864211e-02,
         0.0000000e+00,  0.0000000e+00, -6.3252263e-02,  1.2484924e+01,
         3.0672678e-01,  3.5169701e+01,  3.6954346e+01, -2.3893514e-01,
        -2.4189419e-01, -9.9883099e+00,  2.7890164e-01, -5.0684112e-01,
         1.1243894e+00, -2.1583936e+00, -2.6389625e+00,  8.3879083e-02,
        -4.7052544e-01, -5.0515455e-01,  3.1640894e+01,  5.9351612e+01,
        -2.5253636e-01, -2.5093088e-01,  0.0000000e+00,  1.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.00000

Test y Set:


array([1, 1, 1, 0, 1])

X_train shape: (61482, 118)
X_test shape: (84104, 118)
X_train dtype: float32
X_test dtype: float32
y_train shape: (61482,)
y_test shape: (84104,)
Training labels:
(array([0]), array([61482]))
Testing labels:
(array([0, 1]), array([26350, 57754]))


## Hyperparameter Grid Setup (3.1)

In [33]:
from sklearn.model_selection import train_test_split
import itertools
import random

# --------------------------------------------------------------------------
# Train/validation split
#
# IMPORTANT: We split a validation set OUT of X_train here rather than using
# X_test for model selection. Using X_test during the hyperparameter search
# (as the original boilerplate grid-search cell did via validation_data=
# (X_test, y_test)) leaks test-set information into model selection, so any
# "best model" chosen that way would have an optimistic, invalid final score.
# X_test/y_test should only be touched once, in the final evaluation section.
# --------------------------------------------------------------------------
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Fit set:       {X_train_fit.shape}")
print(f"Validation set:{X_val.shape}")

# --------------------------------------------------------------------------
# Hyperparameter search space
#
# latent_dim     - size of the autoencoder bottleneck (the key architecture
#                  knob for anomaly detection: too large -> reconstructs
#                  anomalies too well; too small -> can't reconstruct normal
#                  data well either)
# learning_rate  - Adam step size
# batch_size     - training batch size
# dropout_rate   - dropout after each hidden layer (0.0 = disabled)
# n_hidden_layers- number of hidden layers on EACH side of the bottleneck
#                  (encoder and decoder are mirrored), i.e. "depth"
# --------------------------------------------------------------------------
search_space = {
    'latent_dim': [8, 16, 32, 64],
    'learning_rate': [1e-2, 1e-3, 1e-4],
    'batch_size': [32, 64, 128],
    'dropout_rate': [0.0, 0.1, 0.2, 0.3],
    'n_hidden_layers': [1, 2, 3],
}

n_total_combos = 1
for v in search_space.values():
    n_total_combos *= len(v)
print(f"Full grid size: {n_total_combos} combinations")


def sample_random_configs(space, n_samples, seed=42):
    """
    Randomly sample n_samples unique hyperparameter configurations from the
    search space. We use random search instead of exhaustive grid search:
    with a large number of combinations, a full grid is not tractable to train in a
    course-project timeframe, and random search is known to find
    comparably good configurations with far fewer trials.
    """
    rng = random.Random(seed)
    keys = list(space.keys())
    all_combos = list(itertools.product(*space.values()))
    rng.shuffle(all_combos)
    sampled = all_combos[:n_samples]
    return [dict(zip(keys, combo)) for combo in sampled]


N_CONFIGS = 100  # search budget; increase if you have compute/time to spare
configs = sample_random_configs(search_space, N_CONFIGS)
print(f"Sampled {len(configs)} of {n_total_combos} possible configurations")

Fit set:       (49185, 118)
Validation set:(12297, 118)
Full grid size: 432 combinations
Sampled 100 of 432 possible configurations


In [34]:
# Test code cells here
print("Example sampled configs:")
for cfg in configs[:5]:
    print(cfg)

Example sampled configs:
{'latent_dim': 64, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.3, 'n_hidden_layers': 1}
{'latent_dim': 32, 'learning_rate': 0.01, 'batch_size': 64, 'dropout_rate': 0.0, 'n_hidden_layers': 1}
{'latent_dim': 32, 'learning_rate': 0.001, 'batch_size': 128, 'dropout_rate': 0.0, 'n_hidden_layers': 3}
{'latent_dim': 8, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.2, 'n_hidden_layers': 3}
{'latent_dim': 16, 'learning_rate': 0.01, 'batch_size': 128, 'dropout_rate': 0.2, 'n_hidden_layers': 2}


## Model Tuning & Architecture Testing (3.2)

In [35]:
# Develop functions here (Finsihed function go into the .py files in the src folder)
# Develop functions here (Finished functions go into the .py files in the src folder)

from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time

INPUT_DIM = X_train_fit.shape[1]


def build_autoencoder(input_dim, latent_dim, dropout_rate=0.0, n_hidden_layers=1, activation='relu'):
    """
    Build a symmetric (undercomplete) autoencoder.

    Hidden layer widths shrink geometrically from input_dim down to
    latent_dim across n_hidden_layers steps, then mirror back up for the
    decoder. This lets 'n_hidden_layers' act as a single "depth" knob in
    the search space instead of hardcoding fixed layer sizes.
    """
    if n_hidden_layers > 0:
         widths = np.linspace(np.log(input_dim), np.log(max(latent_dim, 2)), n_hidden_layers + 2)
                # cast to native Python int: numpy.int64 fails Keras's `units` validation
                # ("expected a positive integer") even though the value itself is fine
         widths = [int(w) for w in np.exp(widths).astype(int)[1:-1]]
    else:
        widths = []

    model = Sequential(  
        name=f"autoencoder_ld{latent_dim}_L{n_hidden_layers}_do{dropout_rate}"
    )
    model.add(layers.Input(shape=(input_dim,)))

    # Encoder
    for w in widths:
        model.add(layers.Dense(w, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(latent_dim, activation=activation, name="bottleneck"))

    # Decoder (mirror of encoder)
    for w in reversed(widths):
        model.add(layers.Dense(w, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(input_dim, activation='linear'))  # linear output for reconstruction

    return model


def train_and_score_config(config, X_fit, X_val, epochs=50, patience=5, verbose=0):
    """
    Train one autoencoder configuration (reconstruction task: X -> X) and
    return the trained model, its training history, and its best
    validation reconstruction loss (MSE).
    """
    model = build_autoencoder(
        input_dim=INPUT_DIM,
        latent_dim=config['latent_dim'],
        dropout_rate=config['dropout_rate'],
        n_hidden_layers=config['n_hidden_layers'],
    )
    model.compile(optimizer=Adam(learning_rate=config['learning_rate']), loss='mse')

    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    history = model.fit(
        X_fit, X_fit,
        validation_data=(X_val, X_val),
        epochs=epochs,
        batch_size=config['batch_size'],
        callbacks=[early_stop],
        verbose=verbose,
    )

    best_val_loss = float(min(history.history['val_loss']))
    return model, history, best_val_loss

In [ ]:
# Test code cells here
TRAIN_ON_NORMAL_ONLY = True
NORMAL_LABEL = 0

if TRAIN_ON_NORMAL_ONLY:
    normal_mask = (y_train_fit == NORMAL_LABEL)
    X_fit_ae = X_train_fit[normal_mask]
else:
    X_fit_ae = X_train_fit

print(f"Training autoencoder on {X_fit_ae.shape[0]} samples "
      f"({'normal-only' if TRAIN_ON_NORMAL_ONLY else 'all training data'})")

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

results = []
best_val_loss = np.inf
best_model = None
best_config = None

for i, config in enumerate(configs):
    start = time.time()
    print(f"\n[{i+1}/{len(configs)}] Training config: {config}")

    try:
        model, history, val_loss = train_and_score_config(
            config, X_fit_ae, X_val, epochs=50, patience=5
        )
    except Exception as e:
        print(f"  Failed: {e}")
        continue

    elapsed = time.time() - start
    print(f"  val_loss={val_loss:.6f}  ({elapsed:.1f}s, {len(history.history['loss'])} epochs run)")

    results.append({**config, 'val_loss': val_loss, 'train_time_s': elapsed})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model
        best_config = config
        best_model.save(models_dir / "autoencoder_model_best.keras")
        print(f"  --> New best model saved (val_loss={val_loss:.6f})")

if not results:
    print("\nAll configurations failed to train -- check the error messages above "
          "before proceeding (results_df would be empty).")
else:
    results_df = pd.DataFrame(results).sort_values('val_loss').reset_index(drop=True)
    results_df.to_csv(models_dir / "hyperparam_search_results.csv", index=False)

    print("\nTop 10 configurations by validation loss:")
    display(results_df.head(10))
    print(f"\nBest config: {best_config}")
    print(f"Best validation loss: {best_val_loss:.6f}")

Training autoencoder on 49185 samples (normal-only)

[1/100] Training config: {'latent_dim': 64, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.3, 'n_hidden_layers': 1}


In [ ]:
if 'results_df' not in dir() or results_df.empty:
    print("results_df is empty -- run the search cell above first.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Latent dimension vs val_loss, colored by depth, sized by train time
    sns.scatterplot(
        data=results_df, x='latent_dim', y='val_loss', hue='n_hidden_layers',
        size='train_time_s', palette='viridis', ax=axes[0, 0], legend='brief'
    )
    axes[0, 0].set_title('Validation Loss vs Latent Dimension')
    axes[0, 0].set_yscale('log')

    # 2. Learning rate vs val_loss
    sns.boxplot(data=results_df, x='learning_rate', y='val_loss', ax=axes[0, 1])
    axes[0, 1].set_title('Validation Loss by Learning Rate')
    axes[0, 1].set_yscale('log')

    # 3. Batch size vs val_loss
    sns.boxplot(data=results_df, x='batch_size', y='val_loss', ax=axes[0, 2])
    axes[0, 2].set_title('Validation Loss by Batch Size')
    axes[0, 2].set_yscale('log')

    # 4. Dropout rate vs val_loss
    sns.boxplot(data=results_df, x='dropout_rate', y='val_loss', ax=axes[1, 0])
    axes[1, 0].set_title('Validation Loss by Dropout Rate')
    axes[1, 0].set_yscale('log')

    # 5. Network depth vs val_loss
    sns.boxplot(data=results_df, x='n_hidden_layers', y='val_loss', ax=axes[1, 1])
    axes[1, 1].set_title('Validation Loss by Network Depth')
    axes[1, 1].set_yscale('log')

    # 6. Top-10 configurations, ranked
    top_n = results_df.head(10).copy()
    top_n['config_label'] = top_n.apply(
        lambda r: f"ld={int(r.latent_dim)},lr={r.learning_rate},bs={int(r.batch_size)},"
                  f"do={r.dropout_rate},L={int(r.n_hidden_layers)}", axis=1
    )
    sns.barplot(data=top_n, y='config_label', x='val_loss', ax=axes[1, 2], palette='crest')
    axes[1, 2].set_title('Top 10 Configurations')
    axes[1, 2].set_xlabel('Validation Loss')
    axes[1, 2].set_ylabel('')

    plt.tight_layout()
    plt.show()

    # Correlation of each hyperparameter with validation loss
    corr_cols = ['latent_dim', 'learning_rate', 'batch_size', 'dropout_rate', 'n_hidden_layers', 'val_loss']
    corr = results_df[corr_cols].corr()

    plt.figure(figsize=(7, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Hyperparameter Correlation Matrix')
    plt.tight_layout()
    plt.show()

    val_loss_corr = corr['val_loss'].drop('val_loss').sort_values(key=abs, ascending=False)
    print("Correlation of each hyperparameter with validation loss (sorted by strength):")
    print(val_loss_corr)

## Evalute Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize Global Variables for model development
THRESHOLD_PERCENTILE = 95

def calculate_reconstruction_error(model, data):
    """Calculate reconstruction error (MSE) for each sample."""
    reconstructed_data = model.predict(data, verbose=0)
    return np.mean(np.square(data - reconstructed_data), axis=1)

def generate_anomaly_metrics_and_threshold(model, X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE):
    """
    Generate anomaly threshold and evaluation metrics with visualization.
    
    Parameters:
    - model: Trained autoencoder model
    - X_train: Training data (for threshold calibration)
    - X_test: Test data (for evaluation)
    - y_test: Test labels
    - percentile: Percentile for threshold determination
    
    Returns:
    - threshold: Calculated anomaly threshold
    - y_pred: Predicted anomaly labels for test set
    """
    # Calculate reconstruction errors (single pass)
    train_errors = calculate_reconstruction_error(model, X_train)
    test_errors = calculate_reconstruction_error(model, X_test)
    
    # Determine threshold from training data
    threshold = np.percentile(train_errors, percentile)
    print(f"Train set Anomaly Threshold (Percentile {percentile}): {threshold:.4f}\n")
    
    # Predict anomalies for test set
    y_pred = np.where(test_errors > threshold, 1, 0)

    # Visualization 1: Reconstruction Error Distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.histplot(train_errors, bins=50, kde=True, label='Training', alpha=0.7)
    plt.axvline(threshold, color='r', linestyle='--', linewidth=2, label=f'Threshold ({percentile}th %ile)')
    plt.title('Reconstruction Error Distribution (Training Data)')
    plt.xlabel('Reconstruction Error')
    plt.ylabel('Frequency')
    plt.legend()
    
    # Visualization 2: Test Set Anomaly Detection
    plt.subplot(1, 2, 2)
    colors = ['green' if pred == 0 else 'red' for pred in y_pred]
    plt.scatter(range(len(test_errors)), test_errors, c=colors, alpha=0.6, s=30)
    plt.axhline(threshold, color='r', linestyle='--', linewidth=2, label='Threshold')
    plt.title('Test Set: Reconstruction Error vs Sample Index')
    plt.xlabel('Sample Index')
    plt.ylabel('Reconstruction Error')
    plt.legend()
    
    # plt.tight_layout()
    plt.show()
    
    # Visualization 2: Test Set Anomaly Detection (with Log Scale)
    plt.subplot(1, 2, 2)
    colors = ['green' if pred == 0 else 'red' for pred in y_pred]
    plt.scatter(range(len(test_errors)), test_errors, c=colors, alpha=0.6, s=30)
    plt.axhline(threshold, color='r', linestyle='--', linewidth=2, label='Threshold')
    plt.yscale('log') # <--- Added to handle extreme error scales cleanly
    plt.title('Test Set: Reconstruction Error vs Sample Index (Log Scale)')
    plt.xlabel('Sample Index')
    plt.ylabel('Reconstruction Error (Log Scale)')
    plt.legend()
    
    plt.show()

    # Evaluation Metrics
    print("="*50)
    print("TEST SET EVALUATION METRICS")
    print("="*50)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


    return threshold, y_pred


In [ ]:
print("Generating anomaly metrics and threshold")
# threshold, y_pred = generate_anomaly_metrics_and_threshold(model, X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE)

Generating anomaly metrics and threshold


#### Example Code

In [ ]:
# Load the model (assuming the model is saved in the 'models' directory)
model_path = Path("../models/autoencoder_model.h5")
if model_path.exists():
    model = keras.models.load_model(model_path)
    print("Model loaded successfully.")
else:
    print("Model file not found. Please ensure the model is trained and saved in the 'models' directory.")
    
# Evaluate the model's anomaly detection performance on the test set
# _, _ = generate_anomaly_metrics_and_threshold(..., X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE)



Model file not found. Please ensure the model is trained and saved in the 'models' directory.


In [ ]:
'''# Define the hyperparameter grid
hyperparameter_grid = {
    'learning_rate': [0.001, 0.01, 0.1],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20, 30],
    'activation_function': ['relu', 'sigmoid', 'tanh']
}

# Define the model architecture
def create_model(learning_rate, batch_size, epochs, activation_function):
    # Create the model architecture
    model = Sequential()
    model.add(Dense(64, activation=activation_function, input_shape=(784,)))
    model.add(Dense(32, activation=activation_function))
    model.add(Dense(10, activation='softmax'))
    
    # Compile the model
    model.compile(optimizer=Adam(lr=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Perform grid search
for learning_rate in hyperparameter_grid['learning_rate']:
    for batch_size in hyperparameter_grid['batch_size']:
        for epochs in hyperparameter_grid['epochs']:
            for activation_function in hyperparameter_grid['activation_function']:
                # Create the model
                model = create_model(learning_rate, batch_size, epochs, activation_function)
                
                # Train the model
                model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, y_test))
                
                # Evaluate the model
                loss, accuracy = model.evaluate(X_test, y_test)
                print(f'Learning Rate: {learning_rate}, Batch Size: {batch_size}, Epochs: {epochs}, Activation Function: {activation_function}, Loss: {loss}, Accuracy: {accuracy}')

                '''

"# Define the hyperparameter grid\nhyperparameter_grid = {\n    'learning_rate': [0.001, 0.01, 0.1],\n    'batch_size': [32, 64, 128],\n    'epochs': [10, 20, 30],\n    'activation_function': ['relu', 'sigmoid', 'tanh']\n}\n\n# Define the model architecture\ndef create_model(learning_rate, batch_size, epochs, activation_function):\n    # Create the model architecture\n    model = Sequential()\n    model.add(Dense(64, activation=activation_function, input_shape=(784,)))\n    model.add(Dense(32, activation=activation_function))\n    model.add(Dense(10, activation='softmax'))\n\n    # Compile the model\n    model.compile(optimizer=Adam(lr=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])\n\n    return model\n\n# Perform grid search\nfor learning_rate in hyperparameter_grid['learning_rate']:\n    for batch_size in hyperparameter_grid['batch_size']:\n        for epochs in hyperparameter_grid['epochs']:\n            for activation_function in hyperparameter_grid['activat